# 03 QLoRA Training Pilot\n
Run a short pilot training on a subset to validate configuration.

In [ ]:
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
from datasets import Dataset, load_from_disk

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import PeftModel

import wandb

print("✓ All imports successful!")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
print("="*60)
print("PHASE 3: QLoRA Configuration Setup")
print("="*60)

# Model Configuration
BASE_MODEL = "mistralai/Mistral-7B"
print(f"\nBase Model: {BASE_MODEL}")

# BitsAndBytes Configuration (4-bit quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
print("\n✓ BitsAndBytes Config:")
print(f"  - Load in 4-bit: True")
print(f"  - Quantization type: nf4")
print(f"  - Double quantization: True")

# Load tokenizer
print(f"\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✓ Tokenizer loaded")
print(f"  - Vocab size: {tokenizer.vocab_size}")
print(f"  - Pad token: {tokenizer.pad_token_id}")

In [ ]:
print("\nLoading data...")

# Load training data
data_path = Path("../data/processed")
train_data = []

with open(data_path / "train.jsonl", "r") as f:
    for line in f:
        train_data.append(json.loads(line))

# For pilot: use only first 100 samples
pilot_size = 100
pilot_data = train_data[:pilot_size]

print(f"Loaded {len(train_data)} training samples")
print(f"Using {len(pilot_data)} samples for pilot")

# Create dataset
def format_prompt(sample):
    instruction = sample['instruction']
    context = sample['context']
    output = sample['output']
    
    prompt = f"""[INST] Answer the financial question based on the provided context.

Context: {context}

Question: {instruction} [/INST]

Answer: {output}"""
    
    return {"text": prompt}

# Format and tokenize
formatted_data = [format_prompt(sample) for sample in pilot_data]

def tokenize_function(sample):
    text = sample['text']
    tokenized = tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )
    return tokenized

print("\nTokenizing data...")
tokenized_dataset = Dataset.from_dict({
    "input_ids": [tokenize_function(d)["input_ids"] for d in formatted_data],
    "attention_mask": [tokenize_function(d)["attention_mask"] for d in formatted_data]
})

print(f"✓ Tokenized dataset: {len(tokenized_dataset)} samples")
print(f"  - Input shape: {np.array(tokenized_dataset['input_ids']).shape}")
print(f"  - Attention mask shape: {np.array(tokenized_dataset['attention_mask']).shape}")

## 4. Load and Prepare Data (Pilot Subset)

In [ ]:
print("\nConfiguring LoRA adapters...")

lora_config = LoraConfig(
    r=16,                              # LoRA rank
    lora_alpha=32,                     # LoRA scaling
    target_modules=["q_proj", "v_proj"],  # Query and Value projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✓ LoRA adapter applied")

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable Parameters:")
print(f"  - Trainable: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
print(f"  - Total: {total_params:,}")

## 3. LoRA Configuration

In [ ]:
print("\nLoading base model (4-bit quantized)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("✓ Model loaded successfully")

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
print("✓ Model prepared for k-bit training")

## 2. Load Model and Apply QLoRA

In [ ]:
print("\n" + "="*60)
print("PHASE 4: Pilot Training Setup")
print("="*60)

# Initialize W&B
wandb.init(
    project="finance-rag-assistant",
    name="pilot-training",
    config={
        "model": BASE_MODEL,
        "lora_rank": 16,
        "learning_rate": 2e-4,
        "epochs": 1,
        "batch_size": 2,
        "dataset": "FinQA"
    }
)

# Training arguments (light config for pilot)
training_args = TrainingArguments(
    output_dir="../models/pilot-checkpoint",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    max_grad_norm=0.3,
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    logging_steps=5,
    logging_dir="../models/logs",
    save_strategy="steps",
    save_steps=25,
    eval_strategy="no",  # No evaluation for pilot
    report_to=["wandb"],
    remove_unused_columns=False,
    load_best_model_at_end=False,
    seed=42
)

print("\n✓ Training configuration ready:")
print(f"  - Output dir: {training_args.output_dir}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - Total steps: {len(tokenized_dataset) // training_args.per_device_train_batch_size}")

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    tokenizer=tokenizer
)

print("\n✓ Trainer initialized")
print("Ready to run pilot training!")
print("\nNext: Execute the cell below to start training")

In [ ]:
print("Starting pilot training...")
print("⏱  This may take 5-15 minutes depending on your GPU")

try:
    # Train
    train_result = trainer.train()
    
    print("\n" + "="*60)
    print("✓ TRAINING COMPLETED SUCCESSFULLY!")
    print("="*60)
    print(f"\nTraining Results:")
    print(f"  - Final loss: {train_result.training_loss:.4f}")
    print(f"  - Total steps: {train_result.global_step}")
    print(f"  - Training time: {train_result.training_time_in_s:.1f}s")
    
    # Save adapter
    print(f"\nSaving LoRA adapter...")
    model.save_pretrained("../models/finance-adapter-pilot")
    tokenizer.save_pretrained("../models/finance-adapter-pilot")
    print(f"✓ Saved to: ../models/finance-adapter-pilot")
    
    wandb.finish()
    
except Exception as e:
    print(f"\n✗ Training failed with error:")
    print(f"  {type(e).__name__}: {str(e)}")
    print(f"\nCommon issues:")
    print(f"  - Insufficient GPU memory: Reduce batch size or use gradient checkpointing")
    print(f"  - CPU training: This will be very slow. Consider using a GPU")
    wandb.finish()

## 5. Run Pilot Training